# Notebook 04: Feature Engineering
## Forecasting Education Performance -- Tanzania BEST Datasets (2020-2024)
**Author:** Habil Masawika | **Project:** Tanzania BEST ML Forecasting

---

### Objectives
1. Create lag features (1-year, 2-year) to capture system momentum
2. Construct rolling averages to smooth noisy annual fluctuations
3. Derive year-over-year growth rates for enrolment, teachers, and schools
4. Build composite infrastructure, ICT, and education quality indices
5. Generate interaction features grounded in education production theory
6. Create the target lag feature critical for forecasting
7. Document every feature with rationale and expected direction of effect

In [ ]:
import sys, warnings
sys.path.insert(0, '/home/claude/BEST-ML-Forecasting/src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utilities import setup_logging, set_seeds, ProjectPaths, save_dataframe
from feature_engineering import FeatureEngineer, get_model_features

set_seeds(42)
logger = setup_logging()
paths  = ProjectPaths()

panel = pd.read_csv(paths.processed('best_panel_cleaned.csv'))
print(f"Loaded cleaned panel: {panel.shape}")

In [ ]:
# ── Run feature engineering pipeline ─────────────────────────────────────
fe = FeatureEngineer(panel)
panel_fe = fe.build()

print(f"\nEngineered panel: {panel_fe.shape[0]} rows x {panel_fe.shape[1]} columns")
print(f"New features: {panel_fe.shape[1] - panel.shape[1]}")

In [ ]:
# ── Feature descriptions ──────────────────────────────────────────────────
feat_desc = fe.feature_descriptions()
print(f"\nAll engineered features ({len(feat_desc)}):")
for _, row in feat_desc.iterrows():
    print(f"  {row['feature']:<40} {row['description'][:70]}")

In [ ]:
# ── Feature correlation with target ──────────────────────────────────────
model_feats = get_model_features(panel_fe, target='csee_pass_rate',
                                  include_lags=True, include_interactions=True)
print(f"\nModel features selected: {len(model_feats)}")

target_corr = (panel_fe[model_feats + ['csee_pass_rate']]
               .dropna()
               .corr()['csee_pass_rate']
               .drop('csee_pass_rate')
               .sort_values(key=abs, ascending=False))

print("\nFeature correlations with CSEE pass rate (|r| > 0.2):")
for feat, val in target_corr[target_corr.abs() > 0.2].items():
    bar = '█' * int(abs(val) * 20)
    sign = '+' if val > 0 else '-'
    print(f"  {feat:<40} {sign}{abs(val):.3f}  {bar}")

In [ ]:
# ── Visualise key engineered features ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

features_to_plot = [f for f in [
    'infra_index', 'education_quality_index', 'enrolment_yoy_growth',
    'teacher_quality_x_density', 'retention_pressure', 'csee_pass_rate_lag1'
] if f in panel_fe.columns]

for i, feat in enumerate(features_to_plot[:6]):
    data = panel_fe[feat].dropna()
    axes[i].hist(data, bins=18, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].axvline(data.mean(), color='crimson', lw=2, linestyle='--',
                    label=f'Mean={data.mean():.3f}')
    axes[i].set_title(feat.replace('_', ' ').title(), fontweight='bold', fontsize=10)
    axes[i].legend(fontsize=8)

for j in range(len(features_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Key Engineered Features', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(paths.fig('nb04_engineered_features.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Infrastructure and quality indices ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Infrastructure index by region
reg_infra = panel_fe.groupby('region')['infra_index'].mean().sort_values()
axes[0].barh(reg_infra.index, reg_infra.values, color='mediumpurple', alpha=0.85)
axes[0].set_xlabel('Infrastructure Index (0-1)')
axes[0].set_title('Regional Infrastructure Index
(Average 2020-2024)', fontweight='bold')
axes[0].invert_yaxis()

# Education quality index by year
if 'education_quality_index' in panel_fe.columns:
    yr_eq = panel_fe.groupby('year')['education_quality_index'].mean()
    axes[1].plot(yr_eq.index, yr_eq.values, 'o-', color='teal', lw=2.5, ms=8)
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Education Quality Index')
    axes[1].set_title('National Education Quality Index Trend', fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(paths.fig('nb04_indices.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Growth features ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
years = sorted(panel_fe['year'].unique())[1:]  # skip 2020 (no prior year)
growth_cols = ['enrolment_yoy_growth', 'teacher_yoy_growth', 'school_yoy_growth']
growth_cols = [c for c in growth_cols if c in panel_fe.columns]
colours = ['steelblue', 'darkorange', 'mediumseagreen']

for col, colour in zip(growth_cols, colours):
    yr_g = panel_fe.groupby('year')[col].mean()
    ax.plot(yr_g.index, yr_g.values * 100, 'o-', color=colour, lw=2,
            ms=7, label=col.replace('_yoy_growth','').replace('_',' ').title() + ' Growth %')

ax.axhline(0, color='black', lw=1, linestyle='--')
ax.set_xlabel('Year')
ax.set_ylabel('Average YoY Growth Rate (%)')
ax.set_title('Year-over-Year Growth Rates: Enrolment, Teachers, Schools', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(paths.fig('nb04_growth_rates.png'), dpi=150, bbox_inches='tight')
plt.show()
print('INTERPRETATION: Enrolment growth consistently exceeds teacher growth, quantifying the')
print('structural teacher-supply gap. School construction tracks roughly in line with teacher')
print('recruitment, suggesting that physical infrastructure expansion is not the binding constraint')
print('-- teacher supply and qualification are.')

In [ ]:
# ── Save engineered panel ────────────────────────────────────────────────
save_dataframe(panel_fe, paths.processed('best_panel_features.csv'))
print(f"Engineered panel saved: {panel_fe.shape}")
print(f"Model features ({len(model_feats)}): {model_feats}")